
# 30 组合与最优化：怎样同时管住几种风险

> **本章要解决的问题**：一个组合里既有期货、又有期权，Delta、Vega、Gamma 混在一起，怎样变成一个优化问题？为什么协方差矩阵的一点噪声，会让权重爆炸？整数手数、保证金、换手，怎样进约束？

## 本章导读

第07章讲了一般的最优化，第25章讲了希腊字母。这一章把两者合起来：**把不同来源的风险换算成同一单位，再写成一个带约束的优化问题。**

本章分六节：

1. 风险单位：Delta、Vega、Gamma 怎样变成现金；
2. 目标函数：风险 + 成本 + 惩罚；
3. 协方差收缩：为什么小噪声会被求逆放大；
4. 求解器：SLSQP/约束/可行性；
5. 风险贡献与风险预算；
6. 整数手数、保证金、换手。

### 学习目标

- 能把期权 Delta/Vega 换算成期货手数/现金风险。
- 能写“风险 + 成本 + 惩罚”的目标和硬约束。
- 会用收缩改善协方差条件数。
- 会用 SLSQP 解带约束组合，并诊断可行性。
- 会算风险贡献，并把连续权重落到整数手数。

### 先修

第07章的凸优化/拉格朗日；第25章的希腊字母；第04章的协方差矩阵。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="30 组合与最优化：同时管住几种风险" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>30 组合与最优化：同时管住几种风险｜知识图</title>
<desc>本章主线：Delta、Vega、Gamma 先换成同一现金单位，再把预算、保证金与整数手数写进约束。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a26" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao26" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag26" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">30</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">组合与最优化：同时管住几种风险</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：Delta、Vega、Gamma 先换成同一现金单位，再把预算、保证金与整数手数写进约束。</text>
<rect x="45" y="82" width="385" height="250" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<text x="60" y="102" font-size="10.5" fill="#1B3A5C" font-weight="bold">① 资本权重 ≠ 风险贡献</text>
<text x="110" y="140" font-size="9.5" fill="#6E7B8B">资本权重</text>
<rect x="110" y="150" width="120" height="40" fill="#1B3A5C" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="170" y="167" font-size="8.6" fill="#FFFFFF" text-anchor="middle">股票</text>
<text x="170" y="181" font-size="8" fill="#FFFFFF" text-anchor="middle">40%</text>
<rect x="230" y="150" width="105" height="40" fill="#E6EFF7" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="282.5" y="167" font-size="8.6" fill="#1B3A5C" text-anchor="middle">债券</text>
<text x="282.5" y="181" font-size="8" fill="#1B3A5C" text-anchor="middle">35%</text>
<rect x="335" y="150" width="45" height="40" fill="#E07B39" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="357.5" y="175" font-size="7.6" fill="#FFFFFF" text-anchor="middle">15%</text>
<rect x="380" y="150" width="30" height="40" fill="#6E7B8B" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="395" y="175" font-size="7.6" fill="#FFFFFF" text-anchor="middle">10%</text>
<text x="110" y="240" font-size="9.5" fill="#6E7B8B">风险贡献</text>
<rect x="110" y="250" width="165" height="40" fill="#1B3A5C" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="192.5" y="267" font-size="8.6" fill="#FFFFFF" text-anchor="middle">股票</text>
<text x="192.5" y="281" font-size="8" fill="#FFFFFF" text-anchor="middle">55%</text>
<rect x="275" y="250" width="54" height="40" fill="#E6EFF7" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="302" y="275" font-size="7.6" fill="#1B3A5C" text-anchor="middle">18%</text>
<rect x="329" y="250" width="66" height="40" fill="#E07B39" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="362" y="267" font-size="8.6" fill="#FFFFFF" text-anchor="middle">商品</text>
<text x="362" y="281" font-size="8" fill="#FFFFFF" text-anchor="middle">22%</text>
<rect x="395" y="250" width="15" height="40" fill="#6E7B8B" stroke="#FFFFFF" stroke-width="1" rx="2"/>
<text x="402.5" y="275" font-size="7.6" fill="#FFFFFF" text-anchor="middle">5%</text>
<line x1="185" y1="138" x2="185" y2="296" stroke="#E07B39" stroke-width="1.3" stroke-linecap="round" stroke-dasharray="5 3"/>
<text x="189" y="136" font-size="8.6" fill="#E07B39">等风险 25%</text>
<path d="M 255 200 L 255 240" fill="none" stroke="#6E7B8B" stroke-width="1.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag26)"/>
<text x="263" y="224" font-size="8.8" fill="#6E7B8B">金额一样，风险不一样</text>
<rect x="60" y="306" width="9" height="9" fill="#1B3A5C" rx="2"/>
<text x="74" y="314" font-size="8.6" fill="#6E7B8B">股票</text>
<rect x="152" y="306" width="9" height="9" fill="#E6EFF7" rx="2"/>
<text x="166" y="314" font-size="8.6" fill="#6E7B8B">债券</text>
<rect x="244" y="306" width="9" height="9" fill="#E07B39" rx="2"/>
<text x="258" y="314" font-size="8.6" fill="#6E7B8B">商品</text>
<rect x="336" y="306" width="9" height="9" fill="#6E7B8B" rx="2"/>
<text x="350" y="314" font-size="8.6" fill="#6E7B8B">现金</text>
<text x="60" y="326" font-size="8.6" fill="#6E7B8B">风险预算：让每类资产的风险贡献接近目标</text>
<rect x="455" y="100" width="295" height="52" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="602.5" y="118.4" font-size="11.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">① 统一风险单位</text>
<text x="602.5" y="136.9" font-size="9" fill="#6E7B8B" text-anchor="middle">Δ / Vega → 现金</text>
<rect x="455" y="164" width="295" height="52" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="602.5" y="182.4" font-size="11.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">② 目标函数</text>
<text x="602.5" y="200.9" font-size="9" fill="#6E7B8B" text-anchor="middle">协方差先收缩</text>
<rect x="455" y="228" width="295" height="52" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="602.5" y="246.4" font-size="11.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">③ 硬约束</text>
<text x="602.5" y="264.9" font-size="9" fill="#6E7B8B" text-anchor="middle">预算 / 保证金 / 换手</text>
<rect x="455" y="292" width="295" height="52" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="602.5" y="310.4" font-size="11.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">④ 整数手数</text>
<text x="602.5" y="328.9" font-size="9" fill="#6E7B8B" text-anchor="middle">连续权重的落地</text>
<path d="M 602.5 152 L 602.5 164" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a26)"/>
<path d="M 602.5 216 L 602.5 228" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a26)"/>
<path d="M 602.5 280 L 602.5 292" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a26)"/>
</svg>

> **图**：本章主线：Delta、Vega、Gamma 先换成同一现金单位，再把预算、保证金与整数手数写进约束。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：组合经理、风险预算、期权组合。
- **输入**：各腿 Greeks、协方差、预算、保证金。
- **输出**：权重/手数、风险贡献、约束下的最优组合。
- **在流程的哪一步**：组合构建与风险预算。
- **失败模式**：风险单位不统一、约束不可行、连续权重不取整。
- **最小工作流**：风险单位 → 目标/约束 → 收缩 → 求解 → 风险贡献 → 整数手数。

---
## 1. 风险单位：把不同风险换算成同一单位

### 1.1 三个换算

- 期权 Delta：每单位标的；
- 期货手数：每手乘数；
- Vega：每 1 个波动率点。

要放在一个优化里，先统一成“现金风险”：

$$\text{现金 Delta}=\Delta\times\text{lots}\times\text{mult},$$

$$\text{现金 Vega}=\text{Vega}\times\text{lots}\times\text{mult}\times0.01\quad(\text{每 1 个点}).$$

**公式拆解**
- **符号**：$\Delta$、Vega 为每单位 Greeks，lots 手数，mult 乘数。
- **结构**：现金 Delta$=\Delta\times\text{lots}\times\text{mult}$；现金 Vega 再乘 0.01。
- **代入**：空 3 手、mult$=10$、$\Delta=0.4$：$D=-12$ 元/点；$h=1.2$ 手，取整残余 $-2$ 或 $+8$。
- **边界**：mult$=1$ 时退化为股票；0.01 漏乘则 Vega 放大 100 倍。
- **用途**：统一风险单位、算对冲手数。

### 1.2 手算

空 3 手期权，每手乘数 10，每单位 Delta 0.4：

$$D=-3\times10\times0.4=-12.$$

要 Delta 中性，需要期货多头 $12/\text{mult}_{\text{futures}}$ 手。若期货乘数 10，需要 1.2 手（实际 1 或 2 手，残余风险要记）。

### 1.3 代码：期货手数与残余



In [2]:

for h in [0, 1, 2]:
    residual = -3 * 10 * 0.4 + 10 * h
    print("期货手数", h, "残余现金 Delta", residual)


期货手数 0 残余现金 Delta -12.0
期货手数 1 残余现金 Delta -2.0
期货手数 2 残余现金 Delta 8.0



**读数与应用**：1 手期货残余 −2，2 手残余 +8；没有整数手能完全中性，残余要记录。

**常见错误**：

1. 用“每单位 Delta”直接当手数。
2. 忘记期权乘数。
3. 把 Vega 的“1 个点”当成 1.0。

**停下来想一想**：为什么期权和期货的 Delta 不能直接相加？

<details><summary>对照你的思路</summary>

期权的 Delta 是“每单位标的”，期货的 Delta 是“每手乘数”；单位不同。要先乘手数/乘数，换成“现金 Delta/每点”，才能相加。

</details>

---



## 2. 目标函数：风险 + 成本 + 惩罚

### 2.1 三项

$$\min_h\ \underbrace{\frac12(h-h_0)^T S (h-h_0)}_{\text{风险}}+\underbrace{\lambda|h-h_0|_1}_{\text{成本}}+\underbrace{\mu\,g(h)}_{\text{惩罚}}.$$

- 风险项：偏离目标越远，风险越大；
- 成本项：换手/费用；
- 惩罚项：软约束（比如超过风险限额就罚）。

**公式拆解**
- **符号**：$h$ 手数，$h_0$ 基准，$S$ 风险矩阵，$\lambda$ 成本系数。
- **结构**：目标 $=\frac12(h-h_0)^TS(h-h_0)+\lambda|h-h_0|_1+\mu g(h)$（风险+成本+惩罚）。
- **代入**：目标 2 手、风险 $(h-2)^2$、成本 $\lambda|h|$：$\lambda=0.5$ 选 2 手，$\lambda=2$ 选 1 手。
- **边界**：$\lambda\to\infty$ 时不交易；$S$ 半正定才凸。
- **用途**：把风险、成本、规则写成一个优化。

### 2.2 硬约束 vs 软惩罚

- 硬约束：绝对不能违反（预算、保证金、涨跌停）；
- 软惩罚：尽量少违反（风格暴露、换手）。

硬约束用 `constraints`/`bounds`；软惩罚写进目标。

### 2.3 代码：惩罚函数



In [3]:

for penalty in [0.5, 2.0]:
    scores = [(h - 2) ** 2 + penalty * abs(h) for h in range(3)]
    print("惩罚", penalty, "各手数损失", scores, "选择", min(range(3), key=scores.__getitem__))


惩罚 0.5 各手数损失 [4.0, 1.5, 1.0] 选择 2
惩罚 2.0 各手数损失 [4.0, 3.0, 4.0] 选择 1



**读数与应用**：惩罚小，选 2 手（贴近目标）；惩罚大，选 1 手（省成本）。惩罚系数是“你愿意为省成本牺牲多少目标”的换算。

**常见错误**：

1. 把风险和成本直接相加，单位不一致。
2. 硬约束写成软惩罚，回测“看起来”满足、实盘违反。
3. 惩罚系数拍脑袋，不做敏感性。

**停下来想一想**：为什么“风险平方 + 成本绝对值”不能直接相加？

<details><summary>对照你的思路</summary>

风险平方的单位是“金额²”，成本绝对值的单位是“金额”；量纲不同，系数要承担单位换算。要么统一到金额/效用，要么用风险厌恶系数把风险转成“愿意付的钱”。

</details>

---



## 3. 协方差收缩：为什么小噪声会被求逆放大

### 3.1 病态

两个资产几乎同步，协方差矩阵的最小特征值很小；求逆相当于除以这个小数字，预期收益的一点误差会被放大成很大的对冲头寸。

### 3.2 收缩

把样本协方差向一个稳定目标（如对角矩阵/单位矩阵）拉：

$$S_{\text{shrunk}}=(1-\lambda)S+\lambda F.$$

$\lambda$ 越大，越稳，但偏差越大。

**公式拆解**
- **符号**：$S$ 样本协方差，$F$ 收缩目标，$\lambda\in[0,1]$ 为收缩强度。
- **结构**：$S_{\text{shrunk}}=(1-\lambda)S+\lambda F$，压非对角元、抬高小特征值。
- **代入**：两资产相关系数 0.99、$F=I$：$\lambda=0$ 条件数 199、权重 $\pm1$；$\lambda=0.5$ 条件数 3、权重 $\pm0.02$。
- **边界**：$\lambda=0$ 病态、$\lambda=1$ 丢相关结构；收缩只稳求逆。
- **用途**：组合优化、求逆前预处理；用偏差换方差。

### 3.3 代码：收缩与权重



In [4]:

import numpy as np

S = np.array([[1.0, 0.99], [0.99, 1.0]])
signal = np.array([0.01, -0.01])

for shrink in [0.0, 0.5, 1.0]:
    stable = (1 - shrink) * S + shrink * np.eye(2)
    weights = np.linalg.solve(stable, signal)
    print("收缩", shrink, "条件数", round(np.linalg.cond(stable), 1), "权重", weights.round(3))


收缩 0.0 条件数 199.0 权重 [ 1. -1.]
收缩 0.5 条件数 3.0 权重 [ 0.02 -0.02]
收缩 1.0 条件数 1.0 权重 [ 0.01 -0.01]



**读数与应用**：不收缩时权重很大；收缩 0.5 后条件数下降、权重变小。收缩是用偏差换方差。

**常见错误**：

1. 对原始协方差直接求逆，不检查条件数。
2. 收缩目标选错（比如把相关结构全抹掉）。
3. 用全样本估计协方差，再回测同一段。

**停下来想一想**：为什么“小特征值方向”的权重容易爆炸？

<details><summary>对照你的思路</summary>

求逆时，小特征值变成大倒数；预期收益的微小误差投影到这个方向，会被放大成很大的权重。这个方向通常对应“多空价差”，也是估计误差最大的方向。

</details>

---



## 4. 求解器：SLSQP、约束、可行性

### 4.1 问题

$$\min_w\ w^TSw\quad\text{s.t.}\quad \mathbf1^Tw=1,\quad w^T\mu\ge r_{\min},\quad 0\le w\le1.$$

- 目标：组合方差；
- 约束：预算、最低收益、上下界。

### 4.2 SLSQP

`scipy.optimize.minimize` 的 SLSQP 适合带等式/不等式约束的小规模问题；大规模/凸问题可用 cvxpy/内点法。

### 4.3 可行性

求解前先检查：

- 约束是否互相矛盾（比如要求收益超过最高资产收益）；
- 初值是否可行；
- 边界是否允许。

### 4.4 代码：三资产组合



In [5]:

import numpy as np
from scipy.optimize import minimize

mu = np.array([0.04, 0.08, 0.12])
cov = np.diag([0.05 ** 2, 0.12 ** 2, 0.25 ** 2])

res = minimize(
    lambda w: w @ cov @ w,
    x0=np.array([1 / 3, 1 / 3, 1 / 3]),
    bounds=[(0, 1)] * 3,
    constraints=[
        {"type": "eq", "fun": lambda w: w.sum() - 1},
        {"type": "ineq", "fun": lambda w: w @ mu - 0.075},
    ],
    method="SLSQP",
)
print("成功", res.success)
print("权重", res.x.round(3), "收益", round(res.x @ mu, 4), "波动", round(np.sqrt(res.fun), 4))


成功 True
权重 [0.328 0.469 0.203] 收益 0.075 波动 0.0775



**读数与应用**：最低收益 7.5% 的约束下，优化器会在 8%/12% 资产之间分配。若把目标收益提到 13%，约束不可行，求解会失败。

**常见错误**：

1. 初值不可行，求解器报错。
2. 约束写反（`ineq` 要求 $\ge0$）。
3. 目标收益超出资产范围，硬解。

**停下来想一想**：为什么“目标收益 13%”不可行？

<details><summary>对照你的思路</summary>

最高资产收益 12%，组合收益是各资产收益的加权平均，不可能超过 12%；要求 13% 的可行集为空。先检查资产收益范围，再设约束。

</details>

---



## 5. 风险贡献与风险预算

### 5.1 边际贡献

组合方差 $w^TSw$ 对权重 $w_i$ 的边际贡献：

$$\frac{\partial\sigma^2}{\partial w_i}=2(Sw)_i.$$

风险贡献占比：

$$RC_i=\frac{w_i(Sw)_i}{w^TSw},\qquad\sum_iRC_i=1.$$

**公式拆解**
- **符号**：$w$ 权重，$S$ 协方差，$w^TSw=\sigma^2$，$RC_i$ 风险贡献占比。
- **结构**：$RC_i=w_i(Sw)_i/(w^TSw)$，各项之和为 1。
- **代入**：$w=(0.328,0.469,0.203)$、对角 cov：$RC\approx(4.5\%,52.7\%,42.8\%)$，20% 权重贡献 42.8% 风险。
- **边界**：$w^TSw=0$ 时无定义；对冲仓位 $RC$ 可为负，和仍为 1。
- **用途**：风险预算、风险平价、归因；权重不等于风险贡献。

### 5.2 风险预算

不是“按权重分配”，而是“按风险贡献分配”。一个 10% 的仓位可能贡献 50% 的风险。

### 5.3 代码：风险贡献



In [6]:

w = res.x
port_var = w @ cov @ w
rc = w * (cov @ w) / port_var

print("权重", w.round(3))
print("风险贡献占比", rc.round(3), "合计", round(rc.sum(), 3))


权重 [0.328 0.469 0.203]
风险贡献占比 [0.045 0.527 0.428] 合计 1.0



**读数与应用**：风险贡献占比之和为 1；它可能和权重差很多。风险预算就是给每个资产/因子设一个 $RC_i$ 上限。

**常见错误**：

1. 把权重当风险贡献。
2. 忽略相关性；两个低波动资产高相关，也会贡献很多风险。
3. 只做一次优化，不做滚动/压力测试。

**停下来想一想**：为什么“等权”不等于“等风险”？

<details><summary>对照你的思路</summary>

等权只让资金相同；波动大/相关性高的资产贡献更多风险。等风险（risk parity）要让 $w_i(Sw)_i$ 相等，权重通常与波动率成反比。

</details>

---



## 6. 整数手数、保证金、换手

### 6.1 整数手数

连续权重 $w_i$ 换成手数：

$$\text{lots}_i=\left\lfloor\frac{w_i\times\text{capital}}{\text{price}_i\times\text{mult}_i}\right\rfloor.$$

取整后，实际权重偏离目标；要重新检查预算/风险/保证金。

### 6.2 保证金

期货/期权卖方要保证金；保证金占用会限制可开手数。优化里要加“保证金 ≤ 可用资金”的约束。

### 6.3 换手

调仓有成本；目标里加换手惩罚，或加“换手 ≤ 上限”的硬约束。

### 6.4 代码：整数手数



In [7]:

capital = 100000
prices = np.array([3000.0, 8000.0, 5000.0])
mult = 10
lots = np.floor(w * capital / (prices * mult)).astype(int)
used = lots * prices * mult

print("目标权重", w.round(3))
print("整数手数", lots)
print("实际占用", used, "合计", used.sum())


目标权重 [0.328 0.469 0.203]
整数手数 [1 0 0]
实际占用 [30000.     0.     0.] 合计 30000.0



**读数与应用**：取整后实际占用可能远低于本金；剩余现金、保证金、风险都要重算。小账户尤其明显：1 手都开不了。

**常见错误**：

1. 连续权重直接当手数。
2. 取整后不检查预算/风险。
3. 忽略保证金，满仓开。

**停下来想一想**：为什么小账户的“理论最优”常常不可执行？

<details><summary>对照你的思路</summary>

一手合约的名义金额/保证金可能就超过账户资金；连续权重解出来 0.3 手，实盘只能 0 或 1 手。取整后权重偏离、风险集中，理论最优失效。

</details>

---



## 本章小结

### 一张表

| 概念 | 公式/做法 | 用途 |
|---|---|---|
| 现金 Delta | $\Delta\times\text{lots}\times\text{mult}$ | 统一风险 |
| 目标 | 风险+成本+惩罚 | 优化 |
| 收缩 | $(1-\lambda)S+\lambda F$ | 病态 |
| SLSQP | 约束/边界 | 求解 |
| 风险贡献 | $w_i(Sw)_i/(w^TSw)$ | 风险预算 |
| 整数手数 | $\lfloor wC/(P\times\text{mult})\rfloor$ | 可执行 |

### 八句话

1. 不同风险先换成现金，再相加。
2. 硬约束用约束，软约束用惩罚。
3. 风险和成本的单位要统一。
4. 协方差小特征值方向容易爆炸。
5. 收缩用偏差换方差。
6. 约束不可行时，先检查资产范围。
7. 等权不等于等风险。
8. 连续权重必须落到整数手数并复查。

---

## 本章习题

**1.** 空 3 手期权，乘数 10，Delta 0.4。现金 Delta 多少？期货乘数 10，需多少手对冲？

**2.** 风险 1、成本 0.5，能直接相加吗？为什么？

**3.** 协方差条件数 200，收缩后 10。收缩的作用是什么？

**4.** 目标收益 13%，最高资产收益 12%。可行吗？

**5.** 权重 (0.5,0.5)，$S=\begin{pmatrix}4&2\\2&9\end{pmatrix}$。求风险贡献占比。

**6.** 本金 10 万，价格 3000，乘数 10，目标权重 0.3。能开几手？

**7.** 为什么等权不等于等风险？

**8.** 为什么取整后要重新检查约束？

<details><summary>参考答案</summary>

**1.** $-3\times10\times0.4=-12$；期货需 $12/10=1.2$ 手（实际 1 或 2 手）。

**2.** 不能；风险平方是“金额²”，成本是“金额”，量纲不同，要系数换算。

**3.** 降低求逆放大，权重更稳；用偏差换方差。

**4.** 不可行；组合收益不可能超过最高资产收益。

**5.** $Sw=(4,5.5)$；$w^TSw=4.75$；贡献 (2,2.75)，占比 42.1%/57.9%。

**6.** $0.3\times100000/(3000\times10)=1$ 手。

**7.** 波动大/相关性高的资产贡献更多风险；等权只让资金相同。

**8.** 取整改变权重，可能违反预算/风险/保证金；要重算。

</details>

---

## 参考资料

- Boyd, S. & Vandenberghe, L. *Convex Optimization*。
- Grinold, R. C. & Kahn, R. N. *Active Portfolio Management*。
